# 🏭 Project 13: Visual Quality Control on Casting Defect Dataset
## 📓 Notebook 2: Baseline CNN & Phân Tích Learning Curves (Tuần 2)

> **Theo đúng Hướng dẫn chi tiết (huongdan.md — Mục 3.1 — 15 điểm Rubric):**
> 1. Xây dựng mô hình CNN thuần túy từ đầu (Baseline Sequential với 3 lớp Conv2D: 32 -> 64 -> 128 -> Flatten -> Dense 128 -> Dropout 0.5 -> Dense 1 sigmoid).
> 2. Huấn luyện mô hình 20–25 epochs với Adam optimizer và Binary Crossentropy.
> 3. Vẽ đồ thị Learning Curves (Loss & Accuracy trên Train/Val) theo từng epoch.
> 4. Phân tích hiện tượng Overfitting.

### 📌 Ô 1: Import thư viện và chuẩn bị dữ liệu (Train / Val / Test)

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Kiểm tra GPU
print("TensorFlow version:", tf.__version__)
print("GPU khả dụng:", tf.config.list_physical_devices('GPU'))

# Tự động tìm thư mục data
base_path = "data"
for root, dirs, files_list in os.walk("data"):
    if any("def_front" in d for d in dirs) and any("ok_front" in d for d in dirs):
        base_path = root
        break

# Thu thập danh sách file
def_files = glob.glob(os.path.join(base_path, "**/*def_front*/**/*.jpeg"), recursive=True) + \
            glob.glob(os.path.join(base_path, "**/*def_front*/**/*.jpg"), recursive=True)
ok_files = glob.glob(os.path.join(base_path, "**/*ok_front*/**/*.jpeg"), recursive=True) + \
           glob.glob(os.path.join(base_path, "**/*ok_front*/**/*.jpg"), recursive=True)

all_files = def_files + ok_files
all_labels = [1] * len(def_files) + [0] * len(ok_files) # 1: Defect, 0: OK

# Chia tập: 70% Train, 15% Val, 15% Test
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_files, all_labels, test_size=0.3, stratify=all_labels, random_state=42
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Tập Train:      {len(train_paths)} ảnh")
print(f"Tập Validation: {len(val_paths)} ảnh")
print(f"Tập Test:       {len(test_paths)} ảnh")

# Tạo tf.data pipeline
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, tf.cast(label, tf.float32)

# Data Augmentation cho tập Train
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.08),
    layers.RandomZoom((-0.1, 0.1)),
    layers.RandomContrast(0.1)
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(len(train_paths), seed=42).map(load_img, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).map(lambda x, y: (data_augmentation(x, training=True), y)).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(load_img, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(load_img, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Đã chuẩn bị xong pipeline dữ liệu!")

### 📌 Ô 2: Xây dựng Kiến trúc Baseline CNN (Theo đúng huongdan.md — Mục 3.1)

In [ ]:
from tensorflow.keras import layers, models

# Kiến trúc chuẩn theo đúng hướng dẫn đề tài
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Ngăn chặn overfitting
    layers.Dense(1, activation='sigmoid') # Phân loại nhị phân
])

model.summary()

### 📌 Ô 3: Compile và Huấn luyện mô hình (20–25 epochs)

In [ ]:
# Compile theo đúng huongdan.md
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')]
)

# Huấn luyện 25 epochs
EPOCHS = 25
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

### 📌 Ô 4: Vẽ đồ thị Learning Curves (Loss & Accuracy trên Train vs Val)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Biểu đồ Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, 'b-o', label='Training Loss')
plt.plot(epochs_range, val_loss, 'r--s', label='Validation Loss')
plt.title('Đồ thị Loss (Mất mát)', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Binary Crossentropy Loss')
plt.legend()
plt.grid(True)

# Biểu đồ Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, 'b-o', label='Training Accuracy')
plt.plot(epochs_range, val_acc, 'g--^', label='Validation Accuracy')
plt.title('Đồ thị Accuracy (Độ chính xác)', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/baseline_cnn_learning_curves.png", dpi=300)
plt.show()

### 📌 Ô 5: Đánh giá mô hình trên tập Test độc lập

In [ ]:
test_results = model.evaluate(test_ds)
print(f"Test Loss:      {test_results[0]:.4f}")
print(f"Test Accuracy:  {test_results[1]*100:.2f}%")
print(f"Test Recall:    {test_results[2]*100:.2f}%")
print(f"Test Precision: {test_results[3]*100:.2f}%")